# Investigação da *baseline* de inferência e estudo de caso

## Introdução

Este *notebook* investiga a *baseline* de inferência implementada no *script* `src/03_cpu_gpu_baseline.py`, tomando como referência a revisão identificada na tabela de informações sobre a *baseline*.

• **Objetivo da investigação**

O objetivo é compreender o funcionamento do código, analisar as escolhas de implementação e os resultados de desempenho, além de reunir evidências que fundamentem decisões técnicas futuras. O documento também contempla um estudo de caso voltado à avaliação da capacidade de processamento e dos custos de infraestrutura para atender à demanda diária de um cliente.

• **Seções**

A investigação é estruturada em **oito seções**:

Nas **seções 1 a 4**, o foco é aplicado sobre o funcionamento da inferência e inclui a reconstrução do fluxo de execução, análise dos parâmetros de geração, investigação do aquecimento (*warmup*) e da metodologia de cronometragem, além do processo de obtenção, armazenamento e carregamento do modelo.

As **seções 5 e 6** tratam sobre memória e desempenho, apresentando a interpretação das métricas de consumo de RAM e VRAM, cálculo dos indicadores de desempenho (*throughput* e *speedup*) e comparação entre as três configurações de execução da inferência.

Por fim, as **seções 7 e 8** focam em reprodutibilidade e a aplicação prática, onde se examina o aprimoramento da persistência e a rastreabilidade dos resultados e um estudo de caso que avalia a viabilidade técnica e econômica do processamento diário de 10 mil clientes, a partir de um *benchmark* com 24 amostras.

• **Subseções**

Cada subseção apresenta a questão técnica que está sendo investigada, seguida pela apresentação dos conceitos, componentes do código envolvidos, as referências consultadas e as verificações realizadas. Quando aplicável, também são registradas as hipóteses formuladas antes dos experimentos, as evidências obtidas, a interpretação dos resultados e as decisões tomadas decorrentes da análise.

Também são documentadas as limitações das conclusões, as dúvidas ainda não esclarecidas e as eventuais alterações realizadas no código, distinguindo os **resultados originalmente publicados** daqueles produzidos durante esta investigação. Dessa forma, busca-se preservar a rastreabilidade do estudo e facilitar sua consulta, reprodução e revisão futura.

## Registros e controle de alterações

• ***Commit* mais recente do repositório no início da investigação**

[9a8ba49810d4b162f9eee03577720f0920ba8a06](https://github.com/leandrocsiqueira/ai-coding-tokenomics/commit/9a8ba49810d4b162f9eee03577720f0920ba8a06)

• **Estado do repositório local durante a revisão da seção 1**

A primeira versão deste *notebook* foi publicada no **GitHub** antes da inclusão deste registro de diagnóstico do **Git**. Consequentemente, a saída apresentada abaixo não representa necessariamente o estado do repositório no início da investigação.

O registro foi incorporado durante o refinamento da seção 1 para documentar a *branch*, o último *commit* e as alterações locais existentes no momento da execução da célula.

A revisão-base utilizada para estudar o funcionamento da inferência permanece identificada pelo *commit* 3a1be993041ed75f77f26f23b0cacf0aff507d30. As modificações realizadas posteriormente no *notebook* fazem parte da evolução do material de estudo.

In [4]:
import subprocess
from datetime import datetime
from pathlib import Path

raiz = subprocess.run(
    ["git", "rev-parse", "--show-toplevel"], check=False, capture_output=True, text=True
)

if raiz.returncode != 0:
    print("Nenhum repositório Git encontrado.")
else:
    projeto = Path(raiz.stdout.strip()).name

    branch = subprocess.run(
        ["git", "branch", "--show-current"], check=False, capture_output=True, text=True
    ).stdout.strip()

    commit = subprocess.run(
        ["git", "log", "-1", "--format=%h - %s"],
        check=False,
        capture_output=True,
        text=True,
    ).stdout.strip()

    status = subprocess.run(
        ["git", "-C", raiz.stdout.strip(), "status", "--short"],
        check=False,
        capture_output=True,
        text=True,
    ).stdout.strip()

    print(f"Projeto: {projeto}")
    print(f"Branch: {branch}")
    print(f"Último commit: {commit}")
    print(f"Estado: {'Alterações pendentes' if status else 'Limpo'}")

    if status:
        print(f"\nArquivos alterados:\n{status}")

print(
    "\nRegistro realizado em:",
    datetime.now().astimezone().isoformat(timespec="seconds"),
)


Projeto: ai-coding-tokenomics
Branch: main
Último commit: 9a8ba49 - fix: add PyTorch CUDA index to dependencies
Estado: Alterações pendentes

Arquivos alterados:
M notebooks/03_estudo_inferencia_01c.ipynb
?? notebooks/01c_Inferencia/
?? notebooks/versao_de_trabalho.ipynb

Registro realizado em: 2026-11-21T21:37:45-03:00


**• Informações sobre a *baseline***

| Item | Identificação |
|---|---|
| Repositório | [AI Coding Tokenomics](https://github.com/leandrocsiqueira/ai-coding-tokenomics) |
| *Script* investigado | [src/03_cpu_gpu_baseline.py](https://github.com/leandrocsiqueira/ai-coding-tokenomics/blob/3a1be993041ed75f77f26f23b0cacf0aff507d30/src/03_cpu_gpu_baseline.py) |
| CSV publicado | [results/cpu_gpu_baseline.csv](https://github.com/leandrocsiqueira/ai-coding-tokenomics/blob/3a1be993041ed75f77f26f23b0cacf0aff507d30/results/cpu_gpu_baseline.csv) |
| *Commit* da revisão-base | [3a1be993041ed75f77f26f23b0cacf0aff507d30](https://github.com/leandrocsiqueira/ai-coding-tokenomics/tree/3a1be993041ed75f77f26f23b0cacf0aff507d30) |
| Data de criação da revisão-base | 05 de outubro de 2026 |

• **Arquivos usados no estudo de caso**

| Item | Identificação |
|---|---|
| Caso do cliente | [01c_Inferencia/docs/CASO_CLIENTE_INFRAESTRUTURA_01c.md](./01c_Inferencia/docs/CASO_CLIENTE_INFRAESTRUTURA_01c.md) |
| Guia do *dataset* | [01c_Inferencia/docs/GUIA_DO_DATASET_01c.md](./01c_Inferencia/docs/GUIA_DO_DATASET_01c.md) |
| Localização do *dataset* | [01c_Inferencia/dataset/consumo_cartao_credito_01c.json](./01c_Inferencia/dataset/consumo_cartao_credito_01c.json) |
| *Script* de carregamento do *dataset* | [01c_Inferencia/dataset/loader_dataset.py](./01c_Inferencia/dataset/loader_dataset.py) |
| Referências AWS | [01c_Inferencia/dataset/referencias_aws_2026-10-19.json](./01c_Inferencia/dataset/referencias_aws_2026-10-19.json) |

• **Ambiente usado nessa investigação**

In [5]:
import platform
from datetime import datetime

import psutil
import torch
import transformers

print("=== EXECUÇÃO ===")
print(f"Data e hora: {datetime.now().astimezone():%d/%m/%Y %H:%M:%S %Z}")

print("\n=== HARDWARE E SISTEMA ===")
print(f"Sistema operacional: {platform.platform()}")

with open("/proc/cpuinfo", encoding="utf-8") as arquivo:
    for linha in arquivo:
        if linha.startswith("model name"):
            print(f"Processador: {linha.split(':', 1)[1].strip()}")
            break

print(f"RAM total: {psutil.virtual_memory().total / 1024**3:.2f} GiB")

if torch.cuda.is_available():
    print("\n=== GPU ===")
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(
        f"VRAM total: {torch.cuda.get_device_properties().total_memory / 1024**3:.2f} GiB"
    )
    print(f"CUDA do PyTorch: {torch.version.cuda or 'Não disponível'}")

print("\n=== SOFTWARE ===")
print(f"Python: {platform.python_version()}")
print(f"PyTorch: {torch.__version__}")
print(f"Transformers: {transformers.__version__}")


=== EXECUÇÃO ===
Data e hora: 21/11/2026 21:37:45 -03

=== HARDWARE E SISTEMA ===
Sistema operacional: Linux-7.0.0-43-generic-x86_64-with-glibc2.39
Processador: 12th Gen Intel(R) Core(TM) i7-12800H
RAM total: 31.03 GiB

=== GPU ===
GPU: NVIDIA RTX A3000 12GB Laptop GPU
VRAM total: 11.63 GiB
CUDA do PyTorch: 13.2

=== SOFTWARE ===
Python: 3.12.3
PyTorch: 2.15.0+cu132
Transformers: 5.22.2


• **Registros de alterações**

Nenhuma alteração realizada.

## Seção 1. Fluxo de execução e inferência

**Controles**

* ***Status*:** [x] Concluída sem dúvidas [ ] Concluída com dúvida [ ] Bloqueada

* **Alterações realizadas no código:** Nenhuma alteração é necessária

### 1.1 - Como se dá o fluxo de execução do *script*?

O diagrama a seguir apresenta o fluxo de execução do script, desde a inicialização do programa até o armazenamento dos resultados em arquivo CSV.

A representação identifica as principais funções, suas responsabilidades e as relações entre as etapas de **preparação**, **inferência**, **medição** e **persistência**. Também detalha o funcionamento da função *`run_inference()`*, utilizada nas três configurações previstas: **CPU FP32**, **CUDA FP32** e **CUDA FP16**.

<img src="./01c_Inferencia/images/01c_Inferencia_diagrama_geral.svg">

O fluxo representado evidencia a separação entre a inicialização do programa, a execução das inferências e o registro dos resultados. O *tokenizer* é carregado no escopo global, antes da chamada de *`main()`*, enquanto o modelo é carregado individualmente em cada execução de *`run_inference()`*, conforme o dispositivo e a precisão numérica selecionados. A função também concentra a preparação das entradas, o aquecimento, a geração medida, a coleta das métricas e a liberação dos recursos utilizados.

A execução começa obrigatoriamente pela configuração **CPU/FP32** e, quando **CUDA** está disponível, prossegue com **CUDA/FP32** e **CUDA/FP16**. As métricas retornadas por cada chamada são reunidas em *`main()`* e posteriormente gravadas em um arquivo **CSV**, estabelecendo um procedimento comum para comparar as configurações.

### 1.2 - Onde o *tokenizer* é criado no código?

O diagrama abaixo isola a inicialização do *tokenizer*, destacando que ela ocorre no **escopo global do módulo, antes da chamada a *`main()`***. Também mostra como a mesma instância é reutilizada durante as diferentes execuções de *`run_inference()`*.

<img src="./01c_Inferencia/images/01c_Inferencia_tokenizacao.svg">

O aspecto central é a diferenciação entre criação e uso do *tokenizer*. A chamada *`AutoTokenizer.from_pretrained(MODEL_IDENTIFIER)`* ocorre uma única vez durante a inicialização do módulo, independentemente de *`main()`* ser executada. Posteriormente, as chamadas de *`run_inference()`* reutilizam essa instância para tokenizar o prompt e decodificar os *tokens* gerados.

### 1.3 - Onde o modelo é usado no código e quais as responsabilidades dos demais componentes durante a execução da inferência?

O modelo é efetivamente usado para gerar respostas nas chamadas feitas à função *`generate()`*, presentes em duas funções:

• **`pre_inference_warmup()`:** chama o modelo para realizar uma geração de até 5 novos IDs de *token* com o propósito de aquecimento (*warmup*). A geração é descartada e essa execução não entra na cronometragem.

• **`execute_inference_with_timing()`:** chama o modelo para gerar até 50 novos tokens. Dessa execução, são obtidos a saída usada na avaliação e o tempo decorrido.

As demais partes do programa organizam essa utilização da seguinte forma:

| Etapa                               	| Parte do programa                                           	| Responsabilidade                                                                                                                                      	|
|-------------------------------------	|-------------------------------------------------------------	|-------------------------------------------------------------------------------------------------------------------------------------------------------	|
| Preparação do *tokenizer*             	| *`AutoTokenizer.from_pretrained()`*                             	| Carrega o *tokenizer* usado para converter o *prompt* em entradas numéricas e, posteriormente, converter os IDs da saída em texto.                        	|
| Preparação da memória GPU           	| *`prepare_cuda_memory_for_inference()`*                         	| Executa a coleta de lixo, limpa o *cache* **CUDA** não utilizado e redefine as estatísticas de pico antes do carregamento da configuração GPU.              	|
| Preparação do modelo e das entradas 	| *`run_inference()`*                                             	| Carrega o modelo no *`dtype`* solicitado, move-o para o dispositivo, ativa *`eval()`*, "tokeniza" o *prompt* e coloca os tensores no mesmo dispositivo do modelo. 	|
| Medição de tempo                    	| *`execute_inference_with_timing()`*                             	| Cronometra a geração medida. Em **CUDA**, sincroniza o dispositivo antes de iniciar o cronômetro e depois da geração, antes de encerrá-lo.                	|
| Cálculo das métricas                	| *`run_inference()`* e *`collect_gpu_memory_metrics()`*              	| Calculam *tokens* gerados e *tokens* por segundo; coletam **RSS** do processo e, na **GPU**, nome do dispositivo, pico de memória alocada e memória reservada.          	|
| Apresentação e registro             	| *`print_inference_results()`* e *`save_inference_results_to_csv()`* 	| Exibem métricas e texto no terminal e gravam as métricas no **CSV**. O texto gerado não é salvo no **CSV**.                                                   	|
| Liberação de recursos               	| Bloco *`finally`* de *`run_inference()`*                            	| Exclui referências ao modelo, às entradas e à saída, executa a coleta de lixo e, quando aplicável, limpa o *cache* **CUDA** não utilizado.                  	|

Assim, carregar o modelo não é o mesmo que executar a geração. O carregamento e a preparação acontecem antes; a geração ocorre em *`generate()`*; a interpretação e o registro dos resultados acontecem depois.

### 1.4 - O que entra e sai da função *`generate()` e como IDs se tornam textos*?

Na execução medida, a chamada à função *`generate()`* é feita da seguinte maneira:

```python

# A constante MAX_NEW_TOKENS é igual a 50

generated_ids = inference_model.generate(**input_data, do_sample=False, max_new_tokens=MAX_NEW_TOKENS)

```

**O que entra pela função?** 

O conteúdo de *`PROMPT`* é primeiro convertido em tensores **PyTorch**:

```python

input_data = tokenizer(PROMPT, return_tensors="pt").to(execution_device)

```

O `**` na frente de **`input_data`** realiza o desempacotamento das entradas fornecidas pelo *tokenizer*, incluindo:

• **`input_ids`:** tensor de IDs numéricos dos *tokens* que representam o *prompt*.

• **`attention_mask`:** tensor que indica quais posições da entrada devem ser consideradas pelo modelo.

Esses tensores ficam no mesmo dispositivo do modelo. Os IDs são valores inteiros: a escolha de FP32 ou FP16 para o modelo não transforma os IDs em números de ponto flutuante.

Além das entradas "tokenizadas", o modelo também recebe:

• **A estratégia de geração:** *`do_sample=False`* desativa a amostragem na escolha dos próximos *tokens*.

• **Número máximo de novos *tokens*:** O parâmetro *`max_new_tokens=MAX_NEW_TOKENS`* estabelece como 50 o limite de novos tokens, sem incluir os tokens do *prompt*. Entretanto, a geração pode terminar antes desse limite.


**Nota**

A geração de aquecimento (*warmup*) recebe as mesmas entradas, mas define o parâmetro *`max_new_tokens`* com o valor da constante *`WARMUP_MAX_NEW_TOKENS`* (que é 5).


**O que sai da função?**

No *script*, a função *`generate()`* retorna um tensor de IDs de *tokens*, armazenado em *`generated_ids`*. Para o modelo usado, a sequência retornada contém os *tokens* da entrada seguidos dos novos *tokens*:

```text

generated_ids: [ IDs do prompt | IDs dos tokens gerados ]

```

Como há apenas um *prompt* por chamada, o tensor possui o seguinte formato:

```text

[1, quantidade de tokens de entrada + quantidade de tokens gerados]

```

Por isso, a quantidade de *tokens* novos é calculada desta maneira:

```python

generated_token_count = (generated_ids.shape[-1] - input_token_count)

```

**Como os IDs se tornam texto?**

A conversão para texto acontece separadamente:

```python

inference_result = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

```

Nessa operação:

• *`generated_ids[0]`* seleciona a única sequência do lote de processamento.
• *`tokenizer.decode()`* chama o tokenizer junto ao método *`decode()`* para transformar a sequência de IDs de *token* já produzida em texto legível.
• *`skip_special_tokens=True`* não inclui os *tokens* especiais na representação textual.

Como o *script* decodifica a sequência inteira, o texto exibido inclui a representação textual dos *tokens* do *prompt* e da continuação gerada, e não apenas a resposta nova. A contagem de *`output_tokens`*, por outro lado, considera somente os novos *tokens* acrescentados pela geração.

<img src="./01c_Inferencia/images/01c_Inferencia_entra_sai_modelo.svg">

## Seção 2. Escolhas de geração e controle da execução na inferência

**Controle**

* ***Status*:** [x] Concluída sem dúvidas [ ] Concluída com dúvida [ ] Bloqueada

* **Alterações realizadas no código:** Nenhuma alteração é necessária

### 2.1 - Desativar a amostragem garante respostas corretas ou idênticas entre dispositivos?

Desativar a amostragem com o parâmetro *`do_sample`* configurado como *`False`* significa adotar uma estratégia determinística de seleção dos próximos *tokens*, em vez de utilizar um processo de seleção probabilística. O modelo **Qwen2.5-0.5B-Instruct** possui um arquivo chamado [`generation_config.json`](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct/blob/main/generation_config.json), que contém as configurações padrão de geração de texto. A biblioteca [**Transformers**](https://huggingface.co/docs/transformers/v5.22.2/en/index) carrega as configurações contidas nesse arquivo juntamente com o modelo e as utiliza na chamada à função `generate()` (a menos que sejam sobrescritas por parâmetros definidos explicitamente). Como pode ser observado, a configuração original do modelo vem com a amostragem habilitada:

```json
{
    "bos_token_id": 151643,
    "pad_token_id": 151643,
    "do_sample": true,
    "eos_token_id": [
        151645,
        151643
    ],
    "repetition_penalty": 1.1,
    "temperature": 0.7,
    "top_p": 0.8,
    "top_k": 20,
    "transformers_version": "4.37.0"
}
```

Ao definir `do_sample=False` e mantendo `num_beams=1`, a geração passa a usar a **estratégia de decodificação gulosa** (*greedy decoding*). Nessa estratégia, o modelo seleciona, a cada etapa, o *token* com a maior pontuação após a aplicação dos mecanismos de processamento de pontuações configurados para a geração.

**Nota:**

O parâmetro *`num_beams`* determina o número de sequências candidatas mantidas durante a geração. Seu valor padrão é `1`, indicando que não será utilizada a [busca em feixes](https://www.width.ai/post/what-is-beam-search) (*beam search*). Quando, ao contrário, *`num_beams>1`* e *`do_sample=False`*, o modelo utiliza a beam search, mantendo e avaliando múltiplas sequências candidatas a cada etapa, com base em suas pontuações acumuladas. Outras estratégias de geração estão descritas na [documentação da Hugging Face Transformers](https://huggingface.co/docs/transformers/v5.22.2/en/main_classes/text_generation).

Com `do_sample=False`, os [parâmetros de amostragem](https://ai-tldr.dev/learn/llm-apis/api-basics/sampling-parameters-explained/) (**`temperature`**, **`top_p`** e **`top_k`**) deixam de influenciar a geração, pois seus efeitos estão associados à amostragem probabilística. Entretanto, o [`repetition_penalty`](https://mbrenndoerfer.com/writing/repetition-penalties-language-model-generation) continua sendo aplicado quando configurado com um valor diferente de `1.0`, modificando as pontuações (*logits*) dos *tokens* antes da seleção. Quando seu valor é superior a `1.0`, essa modificação penaliza os *tokens* já presentes na sequência, incluindo os fornecidos no *prompt*, desencorajando sua repetição.

Entretanto, o fato de a geração estar configurada para selecionar o *token* de maior pontuação **não garante que a resposta do modelo esteja correta**, pois isso corresponde a uma estratégia de decodificação, e não a um mecanismo de verificação factual. Da mesma forma, a ausência de amostragem **não garante resultados idênticos entre dispositivos**. Conforme a  [documentação do **PyTorch**](https://docs.pytorch.org/docs/stable/notes/numerical_accuracy.html), operações em ponto flutuante podem produzir resultados numericamente diferentes entre CPU e GPU, mesmo quando as fontes de aleatoriedade são controladas. Essas diferenças podem alterar as pontuações dos *tokens* e, em determinadas situações, modificar a sequência gerada.

### 2.2 O número máximo de **tokens** precisa ser atingido?

O parâmetro *`max_new_tokens`* estabelece o **número máximo de novos *tokens* que o modelo pode gerar**, sem considerar os *tokens* da entrada. Portanto, trata-se de um limite, e não de uma quantidade obrigatória. No caso da geração medida no *script*, por exemplo, o parâmetro é configurado com o valor da constante *`MAX_NEW_TOKENS`*, que é igual a 50. Nesse caso, é permitido ao modelo gerar até 50 novos *tokens*, mas a geração pode terminar antes, caso o modelo produza um *token* especial de fim de sequência (**EOS** - *End of Sequence*).

Esse comportamento está previsto no arquivo [*`generation_config.json`*](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct/blob/main/generation_config.json) do modelo, onde o parâmetro *`eos_token_id`* contém dois identificadores (ambos reconhecidos pela função *`generate()`* como condições de encerramento da geração):

• 151645: correspondente a *`<|im_end|>`*, que sinaliza o encerramento de uma mensagem

• 151643: correspondente a *`<|endoftext|>`*, que representa o fim de um texto.

Como o modelo possui uma arquitetura do tipo [***decoder-only***](https://www.emergentmind.com/topics/decoder-only-large-language-models), ele gera texto dando continuidade à sequência de *tokens* recebida como entrada. Por esse motivo, o resultado retornado por `generate()` contém tanto os *tokens* da entrada (*prompt*) quanto os novos *tokens* produzidos.

Dessa forma, para determinar quantos *tokens* foram efetivamente gerados, é necessário subtrair o comprimento da entrada do comprimento total da sequência retornada. Essa contagem é realizada na função *`run_inference()`* pela expressão:

```python
generated_token_count = generated_ids.shape[-1] - input_token_count
```

Aqui, *`generated_ids.shape[-1]`* representa o comprimento total da sequência retornada pelo modelo, enquanto *`input_token_count`* corresponde ao número de *tokens* da entrada. A diferença fornece a quantidade de novos *tokens* presentes na saída.

### 2.3 Por que usar **eval()** e **torch.inference_mode()** durante a inferência?

O método [`eval()`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.eval) e a classe [`torch.inference_mode()`]((https://docs.pytorch.org/docs/stable/generated/torch.autograd.grad_mode.inference_mode.html)) possuem finalidades diferentes e complementares.

• **`eval()`**: método do **PyTorch** que coloca o modelo em **modo de avaliação**, preparando-o para realizar previsões em vez de treinamento. Essa configuração altera o comportamento de determinadas operações, mas não desativa o [cálculo de gradientes](https://machinelearningmastery.com/gradient-in-machine-learning/), que é o processo de calcular como os parâmetros do modelo devem ser ajustados para reduzir o erro durante o treinamento. Esse método é executado da seguinte forma na função *`run_inference()`*, logo após o carregamento do modelo e sua transferência para o dispositivo de execução (CPU ou GPU):

```python
inference_model.eval()
```

**Nota:** Embora seja adequado definir explicitamente o modo de avaliação, a própria [biblioteca](https://huggingface.co/docs/transformers/main_classes/model#transformers.PreTrainedModel.from_pretrained) já configure esse modo por padrão ao carregar o modelo com *`from_pretrained()`*.

• **`torch.inference_mode()`**: classe do **PyTorch** que implementa um gerenciador de contexto para executar as operações do modelo durante a inferência sem precisar registrar as informações necessárias ao [cálculo de gradientes](https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html).

Durante o treinamento, esses gradientes são usados para determinar como os parâmetros do modelo devem ser ajustados para reduzir o erro das previsões.

Como a inferência usa o modelo para gerar respostas, sem realizar esse processo de aprendizado, o registro dessas informações é desnecessário. Portanto, **dispensar esse registro reduz o trabalho computacional e pode melhorar a eficiência da execução.**

Esse gerenciador de contexto é usado nas duas gerações. O exemplo abaixo apresenta sua utilização na função *`execute_inference_with_timing()`*:

```python
with torch.inference_mode():
    generated_ids = inference_model.generate(
        **input_data,
        do_sample=False,
        max_new_tokens=MAX_NEW_TOKENS
    )
```

### 2.4 Qual é a diferença entre **with** e **for** em Python?

A instrução [*`with`*](https://www.geeksforgeeks.org/python/with-statement-in-python/) permite executar um bloco de código sob condições controladas por um **gerenciador de contexto**, sendo este responsável por realizar as operações necessárias antes da execução do bloco e ao seu encerramento, mesmo que ocorra uma exceção. No *script*, essa utilização pode ser observada na função *`pre_inference_warmup()`*:

```python
def pre_inference_warmup(inference_model, input_data):
    """Run a short warm-up generation before measurement."""

    print("Running warm-up...")

    with torch.inference_mode():
        generated_warmup_ids = inference_model.generate(
            **input_data,
            do_sample=False,
            max_new_tokens=WARMUP_MAX_NEW_TOKENS
        )

    del generated_warmup_ids
```

Como já explicado, `torch.inference_mode()` é um gerenciador de contexto do **PyTorch**. Nesse exemplo, *`with`* permite entrar nesse contexto antes da chamada ao método *`generate()`*, mantendo o modo de inferência ativado durante a geração de aquecimento. Ao sair do bloco, o gerenciador restaura a configuração anterior desse modo. Assim, *`with`* delimita o trecho de código ao qual essa configuração se aplica.

Para ilustrar esse funcionamento de maneira mais simples, o exemplo a seguir utiliza o gerenciador de contexto *`localcontext()`* da biblioteca [*`decimal`*](https://docs.python.org/3/library/decimal.html) do **Python**. Nesse caso, a instrução permite alterar temporariamente a precisão dos cálculos realizados dentro do bloco, sem modificar a configuração utilizada fora dele. A comparação dos resultados antes, durante e depois da execução do bloco permite observar esse comportamento.

In [1]:
from decimal import Decimal, localcontext

print(f"Fora do with: {Decimal(1) / Decimal(3)}\n")

with localcontext() as ctx:
    ctx.prec = 3
    print(f"Dentro do with: {Decimal(1) / Decimal(3)}\n")

print(f"Fora do with: {Decimal(1) / Decimal(3)}")

Fora do with: 0.3333333333333333333333333333

Dentro do with: 0.333

Fora do with: 0.3333333333333333333333333333


Ao contrário do `with`, o `for` não foi criado para gerenciar contextos de execução, mas para executar repetidamente um bloco de código, percorrendo os elementos de um objeto iterável (como listas, textos ou intervalos numéricos), um de cada vez, até que a iteração seja concluída.

No exemplo a seguir, o *`for`* percorre uma lista de notas para calcular e exibir cada valor dobrado, encerrando automaticamente quando não há mais números na lista:

In [2]:
notas = [3, 4, 5]

for nota in notas:
    print(f"Nota dobrada: {nota * 2}")

Nota dobrada: 6
Nota dobrada: 8
Nota dobrada: 10


### 2.5 Experimento

Para verificar se `max_new_tokens` estabelece uma quantidade obrigatória ou apenas um limite máximo de *tokens*, será realizado um experimento com o modelo **Qwen2.5-0.5B-Instruct**. O teste utilizará um *prompt* que solicita uma resposta de apenas uma palavra, comparando duas gerações com limites de 1 e 50 novos *tokens*.

Neste experimento, o *prompt* será preparado com o [**template de conversa**](https://huggingface.co/docs/transformers/chat_templating) do modelo, por meio de `tokenizer.apply_chat_template()`. Esse recurso organiza a entrada no formato esperado pelo modelo para interações conversacionais.

**Diferentemente do experimento, o *script* original não utiliza esse *template***, realizando a "tokenização" diretamente sobre o texto do *prompt*. A utilização do *template* nesta verificação busca favorecer a geração de uma resposta curta e seu encerramento natural.

• **Dúvida:** O parâmetro `max_new_tokens` determina uma quantidade exata de *tokens* ou apenas um limite máximo para a geração?

• **Expectativa registrada antes da execução:** Na primeira geração, espero que o limite de 1 *token* interrompa a resposta. Na segunda, espero que o modelo produza uma resposta curta e encerre a geração com um *token* de fim de sequência (*EOS*), antes de atingir o limite de 50 *tokens*.

• **Condições mantidas:** mesmo modelo, *tokenizer*, *prompt*, template de conversa, CPU, precisão FP32, modo de avaliação e demais configurações de geração, incluindo `repetition_penalty`. A única condição alterada entre as duas execuções será o valor de `max_new_tokens`.

A verificação observará a quantidade de novos *tokens* produzidos e a ocorrência de *EOS*, permitindo identificar se a geração pode terminar antes de atingir o limite configurado.

In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
inference_model = AutoModelForCausalLM.from_pretrained(
    model_id, dtype=torch.float32
).to("cpu")

inference_model.eval()

mensagens = [{"role": "user", "content": "Responda apenas com a palavra azul."}]

input_data = tokenizer.apply_chat_template(
    mensagens, add_generation_prompt=True, return_dict=True, return_tensors="pt"
)

input_tokens = input_data["input_ids"].shape[-1]

eos_ids = inference_model.generation_config.eos_token_id
eos_ids = eos_ids if isinstance(eos_ids, (list, tuple)) else [eos_ids]

for limite in [1, 50]:
    with torch.inference_mode():
        generated_ids = inference_model.generate(
            **input_data, do_sample=False, max_new_tokens=limite
        )

    new_ids = generated_ids[0, input_tokens:]
    quantidade = new_ids.numel()
    encontrou_eos = new_ids[-1].item() in eos_ids

    resposta = tokenizer.decode(new_ids, skip_special_tokens=True)

    print(f"\nLimite: {limite}")
    print(f"Tokens gerados: {quantidade}")
    print(f"EOS encontrado: {encontrou_eos}")
    print(f"Resposta: {resposta!r}")

    assert quantidade <= limite

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 2858.97it/s]



Limite: 1
Tokens gerados: 1
EOS encontrado: False
Resposta: 'Az'

Limite: 50
Tokens gerados: 4
EOS encontrado: True
Resposta: 'Azul.'


**Interpretação:**

Na primeira execução, com *`max_new_tokens=1`*, o modelo produziu apenas um *token*, resultando na resposta incompleta `Az`, sem gerar um *token* de encerramento (**EOS**). A geração foi interrompida ao atingir o limite configurado.

Isso lembra o conceito de que um *token* pode representar um caractere, vários caracteres, uma palavra inteira ou parte de uma palavra. O modelo gerou um único *token*, cuja representação textual é "Az", composta por dois caracteres. Portanto, *`max_new_tokens=1`* foi respeitado.

Na segunda execução, com `max_new_tokens=50`, o modelo gerou quatro *tokens*, incluindo o *token* **EOS**. Como a decodificação utiliza *`skip_special_tokens=True`*, esse *token* não aparece na resposta textual. Caso seja necessário visualizar o **EOS**, basta realizar a decodificação com *`skip_special_tokens=False`*:

In [5]:
resposta_com_eos = tokenizer.decode(
    new_ids,
    skip_special_tokens=False
)

print(resposta_com_eos)

Azul.<|im_end|>


**Conclusão:** O experimento confirmou que *`max_new_tokens`* determina um limite máximo de novos *tokens*, e não uma quantidade obrigatória. A geração pode ser interrompida ao atingir esse limite ou terminar antecipadamente quando o modelo produz um token de encerramento.

## 3. Warmup e Cronometragem

TO-DO

## 4. Localização do modelo e carregamento

TO-DO

## 5. Memórias

TO-DO

## 6. Throughput e speedup

TO-DO

## 7. Preservação dos resultados

TO-DO

## 8. Recomendação para o Cliente

TO-DO

## 9. Fontes consultadas

https://huggingface.co/docs/transformers/main/en/glossary

https://qwen.ai/blog?id=qwen2.5-llm

https://huggingface.co/docs/transformers/v5.20.2/en/main_classes/tokenizer

https://savrn.com/models/qwen2-5-0-5b-instruct

https://savrn.com/models/compare/qwen2-0-5b-vs-qwen2-5-0-5b-instruct

https://llm-explorer.com/list/?ar=Qwen2ForCausalLM

https://deepgenerativemodels.github.io/notes/autoregressive/

https://developer.nvidia.com/blog/llm-benchmarking-fundamental-concepts/

https://machinelearningmastery.com/how-llms-choose-their-words-a-practical-walk-through-of-logits-softmax-and-sampling/

https://machinelearningmastery.com/a-gentle-introduction-to-attention-masking-in-transformer-models/